# Eksperimen Instance Segmentation Indoor

**NRP 041 · Panoptic Indoor Segmentation v5**

Membandingkan YOLOv8n-seg, YOLOv8s-seg, dan Mask R-CNN ResNet-50 FPN pada ukuran 320, 640, dan 960. Setiap konfigurasi memiliki sel training sendiri. Run dimulai dari 640, kemudian 320 dan 960 dalam setiap bagian model.

| Model | Keluarga | Ukuran input | Epoch |
|---|---|---|---|
| YOLOv8n-seg | One-stage | 640, 320, 960 | 30 |
| YOLOv8s-seg | One-stage | 640, 320, 960 | 30 |
| Mask R-CNN ResNet-50 FPN | Two-stage | 640, 320, 960 | 30 |


In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "roboflow", "ultralytics", "pycocotools"])


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


0

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Pilih GPU di pengaturan Accelerator Kaggle, lalu jalankan ulang."
print("GPU:", torch.cuda.get_device_name(0))
from ultralytics import YOLO
import torchvision
print("Torchvision:", torchvision.__version__)

PyTorch: 2.11.0+cu130
CUDA tersedia: True
GPU: Tesla T4
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Torchvision: 0.26.0+cu130


### Unduh dataset resmi v5


In [3]:
from pathlib import Path

from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

WORK_DIR = Path("/kaggle/working")
RUNS_DIR = WORK_DIR / "visi_komputer_041"
rf = Roboflow(api_key="7f7iWApDwJRfabLV1SCX")
project = rf.workspace("panopticindoor").project("panoptic-indoor-segmentation")
dataset = project.version(5).download(
    "coco-segmentation", location=str(WORK_DIR / "panoptic_indoor_coco_v5")
)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/panoptic_indoor_coco_v5 in coco-segmentation:: 100%|██████████| 930/930 [00:00<00:00, 6819.28it/s]


### Periksa split dan anotasi mask


In [4]:
import json
from pathlib import Path

root = Path(dataset.location)

for split in ("train", "valid", "test"):
    path = root / split / "_annotations.coco.json"
    if not path.is_file():
        print(f"{split}: file anotasi belum ditemukan di {path}")
        continue

    data = json.loads(path.read_text())
    masks = sum(bool(a.get("segmentation")) for a in data["annotations"])
    print(f"{split}: {len(data['images'])} gambar, {masks} anotasi dengan mask")

train: 397 gambar, 4479 anotasi dengan mask
valid: 166 gambar, 2022 anotasi dengan mask
test: 359 gambar, 5344 anotasi dengan mask


### Siapkan format YOLO Segmentation

In [5]:
import json
import shutil
from collections import defaultdict
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import yaml
from ultralytics.data.converter import merge_multi_segment

root = Path(dataset.location)
converted = Path(mkdtemp(prefix="panoptic_indoor_yolo_", dir=str(WORK_DIR)))
split_counts = {"train": 397, "valid": 166, "test": 359}

data_by_split = {
    split: json.loads(
        (root / split / "_annotations.coco.json").read_text(encoding="utf-8")
    )
    for split in split_counts
}

categories = sorted(
    data_by_split["train"]["categories"], key=lambda category: category["id"]
)
names = [category["name"] for category in categories]
class_by_name = {name: index for index, name in enumerate(names)}

for split, data in data_by_split.items():
    if len(data["images"]) != split_counts[split]:
        raise ValueError(f"{split}: jumlah gambar berbeda dari split resmi v5.")

    class_ids = {
        category["id"]: class_by_name[category["name"]]
        for category in data["categories"]
    }

    annotations_by_image = defaultdict(list)
    for ann in data["annotations"]:
        if not ann.get("iscrowd"):
            annotations_by_image[ann["image_id"]].append(ann)

    mask_count = 0
    duplicate_count = 0

    for img in data["images"]:
        filename = Path(img["file_name"])
        image_path = converted / "images" / split / filename
        label_path = (converted / "labels" / split / filename).with_suffix(".txt")
        rows = []
        seen = set()

        for ann in annotations_by_image[img["id"]]:
            polygons = ann.get("segmentation")
            if not isinstance(polygons, list) or not polygons or any(
                not isinstance(polygon, list)
                or len(polygon) < 6
                or len(polygon) % 2
                for polygon in polygons
            ):
                raise ValueError(f"{split}/{filename}: mask ID {ann['id']} tidak valid.")

            class_id = class_ids[ann["category_id"]]
            key = (class_id, tuple(tuple(polygon) for polygon in polygons))
            if key in seen:
                duplicate_count += 1
                continue
            seen.add(key)

            if len(polygons) == 1:
                points = np.asarray(polygons[0], dtype=float).reshape(-1, 2)
            else:
                points = np.concatenate(merge_multi_segment(polygons), axis=0)

            points = points / [img["width"], img["height"]]
            if not np.isfinite(points).all() or (points < 0).any() or (points > 1).any():
                raise ValueError(
                    f"{split}/{filename}: koordinat mask ID {ann['id']} di luar gambar."
                )

            coordinates = " ".join(f"{value:.10g}" for value in points.ravel())
            rows.append(f"{class_id} {coordinates}")

        image_path.parent.mkdir(parents=True, exist_ok=True)
        label_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(root / split / filename, image_path)
        label_path.write_text("\n".join(rows), encoding="utf-8")
        mask_count += len(rows)

    print(
        f"{split}: {len(data['images'])} gambar, {mask_count} mask, "
        f"{duplicate_count} duplikat persis dilewati"
    )

yaml_path = converted / "data.yaml"
yaml_path.write_text(
    yaml.safe_dump({
        "path": str(converted),
        "train": "images/train",
        "val": "images/valid",
        "test": "images/test",
        "names": names,
    }, sort_keys=False),
    encoding="utf-8",
)

print(f"Dataset YOLO siap: {yaml_path}")


train: 397 gambar, 4479 mask, 0 duplikat persis dilewati
valid: 166 gambar, 2021 mask, 1 duplikat persis dilewati
test: 359 gambar, 5341 mask, 3 duplikat persis dilewati
Dataset YOLO siap: /kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml


## Fungsi training dan penyimpanan

YOLO memakai batch 4, seed 41, dan pemilihan checkpoint dari fitness validasi bawaan Ultralytics. Mask R-CNN memakai batch fisik 1 dengan akumulasi gradien 4, SGD learning rate 0.005, dan memilih checkpoint berdasarkan COCO mask AP50–95 pada validasi.

In [6]:
import csv
import gc
import importlib.metadata
import random
import time

import torch

assert torch.cuda.is_available(), "Aktifkan GPU Kaggle sebelum menjalankan training."
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Catat versi pustaka dari sesi yang benar-benar menjalankan eksperimen.
versions = {}
for package in ("torch", "torchvision", "ultralytics", "roboflow", "numpy"):
    versions[package] = importlib.metadata.version(package)
(WORK_DIR / "training_environment.json").write_text(json.dumps(versions, indent=2))


def completed_run(name, checkpoint):
    candidates = [RUNS_DIR / name]
    candidates.extend(Path("/kaggle/input").glob(f"**/{name}/training_complete.json"))
    for candidate in candidates:
        directory = candidate.parent if candidate.is_file() else candidate
        marker = directory / "training_complete.json"
        if marker.is_file() and (directory / checkpoint).is_file():
            info = json.loads(marker.read_text())
            if info.get("epochs") == 30:
                print("Hasil training lengkap ditemukan:", directory)
                return directory
    return None


def finish_run(directory, info):
    info["epochs"] = 30
    info["versions"] = versions
    (directory / "training_complete.json").write_text(json.dumps(info, indent=2))
    archive = shutil.make_archive(str(WORK_DIR / directory.name), "zip", root_dir=directory)
    print("ZIP selesai:", archive, flush=True)


In [7]:
def train_yolo_run(weights, size, name):
    previous = completed_run(name, "weights/best.pt")
    if previous is not None:
        return previous

    started = time.time()
    network = YOLO(weights)
    network.train(
        data=str(yaml_path),
        imgsz=size,
        epochs=30,
        patience=0,
        batch=4,
        device=0,
        seed=41,
        project=str(RUNS_DIR),
        name=name,
        workers=2,
    )
    directory = Path(network.trainer.save_dir)
    finish_run(directory, {
        "model": weights, "input_size": size, "batch_size": 4,
        "seed": 41, "seconds": time.time() - started,
        "best_selection": "Ultralytics validation fitness",
    })
    del network
    gc.collect()
    torch.cuda.empty_cache()
    return directory


In [8]:
import contextlib
import copy
import io
import math

from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
from tqdm.auto import tqdm


class IndoorCocoDataset(Dataset):
    def __init__(self, split, class_names):
        self.split = split
        self.image_dir = root / split
        source = json.loads((self.image_dir / "_annotations.coco.json").read_text())
        self.images = source["images"]
        self.foreground_id = {name: i + 1 for i, name in enumerate(class_names)}
        self.category_to_label = {
            cat["id"]: self.foreground_id[cat["name"]]
            for cat in source["categories"]
        }
        self.label_to_category = {value: key for key, value in self.category_to_label.items()}
        self.annotations = defaultdict(list)
        filtered = []
        seen = set()
        for ann in source["annotations"]:
            key = (ann["image_id"], ann["category_id"], json.dumps(ann["segmentation"]))
            if key in seen:
                continue
            seen.add(key)
            filtered.append(ann)
            self.annotations[ann["image_id"]].append(ann)

        source["annotations"] = filtered
        self.coco = COCO()
        self.coco.dataset = copy.deepcopy(source)
        self.coco.dataset.setdefault("info", {})
        self.coco.createIndex()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        info = self.images[index]
        with Image.open(self.image_dir / info["file_name"]) as image:
            image = to_tensor(image.convert("RGB"))
        height, width = image.shape[-2:]
        if (height, width) != (info["height"], info["width"]):
            raise ValueError(f"Ukuran gambar berbeda dari COCO: {info['file_name']}")
        boxes, labels, masks = [], [], []
        for ann in self.annotations[info["id"]]:
            if ann.get("iscrowd"):
                continue
            mask = self.coco.annToMask(ann).astype(np.uint8)
            if not mask.any():
                raise ValueError(f"Mask kosong pada ID {ann['id']}")
            x, y, w, h = ann["bbox"]
            box = [max(0, x), max(0, y), min(width, x + w), min(height, y + h)]
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError(f"BBox tidak valid pada ID {ann['id']}")
            boxes.append(box)
            labels.append(self.category_to_label[ann["category_id"]])
            masks.append(mask)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "masks": torch.from_numpy(np.stack(masks)) if masks else torch.zeros(
                (0, height, width), dtype=torch.uint8
            ),
            "image_id": torch.tensor(info["id"]),
        }
        return image, target


def collate_detection(batch):
    return tuple(zip(*batch))


train_dataset = IndoorCocoDataset("train", names)
valid_dataset = IndoorCocoDataset("valid", names)


def build_maskrcnn(size):
    network = maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,
        min_size=size, max_size=size,
        box_score_thresh=0.001, box_detections_per_img=100,
    )
    box_features = network.roi_heads.box_predictor.cls_score.in_features
    network.roi_heads.box_predictor = FastRCNNPredictor(box_features, len(names) + 1)
    mask_features = network.roi_heads.mask_predictor.conv5_mask.in_channels
    network.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, len(names) + 1)
    return network.cuda()


@torch.inference_mode()
def validation_mask_ap(network):
    network.eval()
    predictions = []
    for image, target in tqdm(valid_dataset, desc="Validasi mask", total=len(valid_dataset)):
        output = network([image.cuda()])[0]
        image_id = int(target["image_id"])
        for label, score, mask in zip(output["labels"], output["scores"], output["masks"]):
            binary = (mask[0] >= 0.5).cpu().numpy().astype(np.uint8)
            if not binary.any():
                continue
            encoded = mask_utils.encode(np.asfortranarray(binary))
            encoded["counts"] = encoded["counts"].decode("ascii")
            predictions.append({
                "image_id": image_id,
                "category_id": valid_dataset.label_to_category[int(label)],
                "segmentation": encoded,
                "score": float(score),
            })
        del output

    if not predictions:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        predicted_coco = valid_dataset.coco.loadRes(predictions)
        evaluation = COCOeval(valid_dataset.coco, predicted_coco, "segm")
        evaluation.params.imgIds = [image["id"] for image in valid_dataset.images]
        evaluation.evaluate()
        evaluation.accumulate()
        evaluation.summarize()
    return float(evaluation.stats[0]), float(evaluation.stats[1])


creating index...
index created!
creating index...
index created!


In [9]:
def train_maskrcnn_run(size):
    name = f"maskrcnn_r50_fpn_{size}_resmi"
    previous = completed_run(name, "best.pth")
    if previous is not None:
        return previous

    random.seed(41)
    np.random.seed(41)
    torch.manual_seed(41)
    torch.cuda.manual_seed_all(41)
    directory = RUNS_DIR / name
    directory.mkdir(parents=True, exist_ok=True)
    network = build_maskrcnn(size)
    optimizer = torch.optim.SGD(
        [p for p in network.parameters() if p.requires_grad],
        lr=0.005, momentum=0.9, weight_decay=0.0005,
    )
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[20, 27], gamma=0.1)
    scaler = torch.amp.GradScaler("cuda")
    generator = torch.Generator().manual_seed(41)
    loader = DataLoader(
        train_dataset, batch_size=1, shuffle=True, num_workers=0,
        collate_fn=collate_detection, generator=generator,
    )
    start_epoch, best_ap, rows = 0, -1.0, []
    checkpoint = directory / "last.pth"
    # Memulihkan run parsial dari Output versi lama yang ditambahkan sebagai Input.
    if not checkpoint.exists():
        candidates = sorted(Path("/kaggle/input").glob(f"**/{name}/last.pth"))
        if candidates:
            checkpoint = candidates[0]
            if (checkpoint.parent / "best.pth").exists():
                shutil.copy2(checkpoint.parent / "best.pth", directory / "best.pth")
    if checkpoint.exists():
        state = torch.load(checkpoint, map_location="cpu", weights_only=False)
        network.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        scaler.load_state_dict(state["scaler"])
        start_epoch, best_ap, rows = state["epoch"], state["best_ap"], state["history"]
        generator.set_state(state["generator_state"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        del state
        print("Lanjut dari epoch:", start_epoch)

    config = {
        "model": "Mask R-CNN ResNet-50 FPN", "input_size": size,
        "resize": "longest edge with aspect ratio preserved",
        "epochs": 30, "seed": 41, "batch_size": 1, "gradient_accumulation": 4,
        "optimizer": "SGD", "learning_rate": 0.005, "milestones": [20, 27],
        "best_selection": "validation COCO mask AP50:95", "versions": versions,
    }
    (directory / "config.json").write_text(json.dumps(config, indent=2))
    (directory / "classes.json").write_text(json.dumps({
        "names": names, "foreground_offset": 1,
        "label_to_coco_category": train_dataset.label_to_category,
    }, indent=2))

    for epoch in range(start_epoch, 30):
        started = time.time()
        network.train()
        optimizer.zero_grad(set_to_none=True)
        total_loss = 0.0
        for step, (images, targets) in enumerate(tqdm(loader, desc=f"{name} epoch {epoch + 1}/30")):
            images = [image.cuda() for image in images]
            targets = [{key: value.cuda() for key, value in target.items()} for target in targets]
            # Kelompok terakhir boleh kurang dari empat gambar.
            group_size = min(4, len(loader) - (step // 4) * 4)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                losses = network(images, targets)
                loss = sum(losses.values())
            if not math.isfinite(float(loss)):
                raise RuntimeError(f"Loss tidak finite: epoch {epoch + 1}, step {step}")
            scaler.scale(loss / group_size).backward()
            if (step + 1) % 4 == 0 or step + 1 == len(loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(network.parameters(), 10)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
            total_loss += float(loss.detach())
        scheduler.step()
        ap, ap50 = validation_mask_ap(network)
        improved = ap > best_ap
        best_ap = max(best_ap, ap)
        rows.append({
            "epoch": epoch + 1, "train_loss": total_loss / len(loader),
            "val_mask_map50_95": ap, "val_mask_map50": ap50,
            "seconds": time.time() - started,
        })
        with (directory / "results.csv").open("w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        state = {
            "epoch": epoch + 1, "model": network.state_dict(),
            "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(), "best_ap": best_ap, "history": rows,
            "config": config, "names": names, "generator_state": generator.get_state(),
            "torch_rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all(),
            "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
        }
        temporary = directory / "last.tmp.pth"
        torch.save(state, temporary)
        temporary.replace(directory / "last.pth")
        if improved:
            shutil.copy2(directory / "last.pth", directory / "best.pth")
        del state
        print(rows[-1], flush=True)

    finish_run(directory, config)
    del network, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()
    return directory


### Run

In [10]:
import pandas as pd
from IPython.display import display

training_summary = {}


def show_training_result(directory, model_name, size):
    directory = Path(directory)
    history = pd.read_csv(directory / "results.csv")
    history.columns = history.columns.str.strip()
    if model_name == "Mask R-CNN ResNet-50 FPN":
        metric_column = "val_mask_map50_95"
        checkpoint = directory / "best.pth"
    else:
        metric_column = "metrics/mAP50-95(M)"
        checkpoint = directory / "weights" / "best.pt"
    record = {
        "Model": model_name,
        "Input": size,
        "Epoch selesai": len(history),
        "Val mask AP50–95 terbaik": float(history[metric_column].max()),
        "Checkpoint": str(checkpoint),
    }
    training_summary[(model_name, size)] = record
    display(pd.DataFrame([record]).style.format({"Val mask AP50–95 terbaik": "{:.4f}"}))
    pd.DataFrame(training_summary.values()).to_csv(
        WORK_DIR / "ringkasan_training.csv", index=False
    )

## YOLOv8n-seg

### YOLOv8n-seg · input 640

In [11]:
run_yolov8n_seg_640 = train_yolo_run("yolov8n-seg.pt", 640, "yolov8n_seg_640_resmi")
show_training_result(run_yolov8n_seg_640, "YOLOv8n-seg", 640)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_640_resmi, nbs=64

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


       2/30      1.96G      1.115       2.71      3.694      1.373          0         15        640: 100% ━━━━━━━━━━━━ 100/100 11.2it/s 8.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 10.1it/s 2.1s
                   all        166       2021      0.282      0.139       0.13     0.0996      0.304      0.137       0.13     0.0913

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
       3/30      1.96G       1.11      2.595      3.288      1.361          0         13        640: 100% ━━━━━━━━━━━━ 100/100 10.5it/s 9.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 10.3it/s 2.0s
                   all        166       2021      0.428       0.15      0.125     0.0916      0.423      0.145      0.123     0.0786

      

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8n-seg,640,30,0.1426,/kaggle/working/visi_komputer_041/yolov8n_seg_640_resmi/weights/best.pt


### YOLOv8n-seg · input 320

In [12]:
run_yolov8n_seg_320 = train_yolo_run("yolov8n-seg.pt", 320, "yolov8n_seg_320_resmi")
show_training_result(run_yolov8n_seg_320, "YOLOv8n-seg", 320)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_320_resmi, nbs=64

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8n-seg,320,30,0.1073,/kaggle/working/visi_komputer_041/yolov8n_seg_320_resmi/weights/best.pt


### YOLOv8n-seg · input 960

In [13]:
run_yolov8n_seg_960 = train_yolo_run("yolov8n-seg.pt", 960, "yolov8n_seg_960_resmi")
show_training_result(run_yolov8n_seg_960, "YOLOv8n-seg", 960)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_seg_960_resmi, nbs=64

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8n-seg,960,30,0.1326,/kaggle/working/visi_komputer_041/yolov8n_seg_960_resmi/weights/best.pt


## YOLOv8s-seg

### YOLOv8s-seg · input 640

In [14]:
run_yolov8s_seg_640 = train_yolo_run("yolov8s-seg.pt", 640, "yolov8s_seg_640_resmi")
show_training_result(run_yolov8s_seg_640, "YOLOv8s-seg", 640)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_seg_640_resmi, nbs=64

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8s-seg,640,30,0.2209,/kaggle/working/visi_komputer_041/yolov8s_seg_640_resmi/weights/best.pt


### YOLOv8s-seg · input 320

In [15]:
run_yolov8s_seg_320 = train_yolo_run("yolov8s-seg.pt", 320, "yolov8s_seg_320_resmi")
show_training_result(run_yolov8s_seg_320, "YOLOv8s-seg", 320)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_seg_320_resmi, nbs=64

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8s-seg,320,30,0.1707,/kaggle/working/visi_komputer_041/yolov8s_seg_320_resmi/weights/best.pt


### YOLOv8s-seg · input 960

In [16]:
run_yolov8s_seg_960 = train_yolo_run("yolov8s-seg.pt", 960, "yolov8s_seg_960_resmi")
show_training_result(run_yolov8s_seg_960, "YOLOv8s-seg", 960)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/panoptic_indoor_yolo_bu6sd9rw/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_seg_960_resmi, nbs=64

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8s-seg,960,30,0.2104,/kaggle/working/visi_komputer_041/yolov8s_seg_960_resmi/weights/best.pt


## Mask R-CNN ResNet-50 FPN

In [17]:
import contextlib
import copy
import io
import math

from PIL import Image
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    maskrcnn_resnet50_fpn, MaskRCNN_ResNet50_FPN_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
from tqdm.auto import tqdm


class IndoorCocoDataset(Dataset):
    def __init__(self, split, class_names):
        self.split = split
        self.image_dir = root / split
        source = json.loads((self.image_dir / "_annotations.coco.json").read_text())
        self.images = source["images"]
        self.foreground_id = {name: i + 1 for i, name in enumerate(class_names)}
        self.category_to_label = {
            cat["id"]: self.foreground_id[cat["name"]]
            for cat in source["categories"]
        }
        self.label_to_category = {value: key for key, value in self.category_to_label.items()}
        self.annotations = defaultdict(list)
        filtered = []
        seen = set()
        for ann in source["annotations"]:
            key = (ann["image_id"], ann["category_id"], json.dumps(ann["segmentation"]))
            if key in seen:
                continue
            seen.add(key)
            filtered.append(ann)
            self.annotations[ann["image_id"]].append(ann)

        source["annotations"] = filtered
        self.coco = COCO()
        self.coco.dataset = copy.deepcopy(source)
        self.coco.dataset.setdefault("info", {})
        self.coco.createIndex()

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        info = self.images[index]
        with Image.open(self.image_dir / info["file_name"]) as image:
            image = to_tensor(image.convert("RGB"))
        height, width = image.shape[-2:]
        scale_x = width / info["width"]
        scale_y = height / info["height"]
        boxes, labels, masks = [], [], []
        for ann in self.annotations[info["id"]]:
            if ann.get("iscrowd"):
                continue
            mask = self.coco.annToMask(ann).astype(np.uint8)
            if mask.shape != (height, width):
                mask = np.asarray(Image.fromarray(mask).resize(
                    (width, height), resample=Image.Resampling.NEAREST
                )).copy()
            if not mask.any():
                raise ValueError(f"Mask kosong pada ID {ann['id']}")
            x, y, w, h = ann["bbox"]
            box = [
                max(0, x * scale_x), max(0, y * scale_y),
                min(width, (x + w) * scale_x), min(height, (y + h) * scale_y),
            ]
            if box[2] <= box[0] or box[3] <= box[1]:
                raise ValueError(f"BBox tidak valid pada ID {ann['id']}")
            boxes.append(box)
            labels.append(self.category_to_label[ann["category_id"]])
            masks.append(mask)

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "masks": torch.from_numpy(np.stack(masks)) if masks else torch.zeros(
                (0, height, width), dtype=torch.uint8
            ),
            "image_id": torch.tensor(info["id"]),
        }
        return image, target


def collate_detection(batch):
    return tuple(zip(*batch))


train_dataset = IndoorCocoDataset("train", names)
valid_dataset = IndoorCocoDataset("valid", names)


def build_maskrcnn(size):
    network = maskrcnn_resnet50_fpn(
        weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT,
        min_size=size, max_size=size,
        box_score_thresh=0.001, box_detections_per_img=100,
    )
    box_features = network.roi_heads.box_predictor.cls_score.in_features
    network.roi_heads.box_predictor = FastRCNNPredictor(box_features, len(names) + 1)
    mask_features = network.roi_heads.mask_predictor.conv5_mask.in_channels
    network.roi_heads.mask_predictor = MaskRCNNPredictor(mask_features, 256, len(names) + 1)
    return network.cuda()


@torch.inference_mode()
def validation_mask_ap(network):
    network.eval()
    predictions = []
    for image, target in tqdm(valid_dataset, desc="Validasi mask", total=len(valid_dataset)):
        output = network([image.cuda()])[0]
        image_id = int(target["image_id"])
        info = valid_dataset.coco.imgs[image_id]
        for label, score, mask in zip(output["labels"], output["scores"], output["masks"]):
            # COCO mengevaluasi mask pada ukuran koordinat anotasi sumber.
            probability = F.interpolate(
                mask.unsqueeze(0), size=(info["height"], info["width"]),
                mode="bilinear", align_corners=False,
            )[0, 0]
            binary = (probability >= 0.5).cpu().numpy().astype(np.uint8)
            if not binary.any():
                continue
            encoded = mask_utils.encode(np.asfortranarray(binary))
            encoded["counts"] = encoded["counts"].decode("ascii")
            predictions.append({
                "image_id": image_id,
                "category_id": valid_dataset.label_to_category[int(label)],
                "segmentation": encoded,
                "score": float(score),
            })
        del output

    if not predictions:
        return 0.0, 0.0
    with contextlib.redirect_stdout(io.StringIO()):
        predicted_coco = valid_dataset.coco.loadRes(predictions)
        evaluation = COCOeval(valid_dataset.coco, predicted_coco, "segm")
        evaluation.params.imgIds = [image["id"] for image in valid_dataset.images]
        evaluation.evaluate()
        evaluation.accumulate()
        evaluation.summarize()
    return float(evaluation.stats[0]), float(evaluation.stats[1])

creating index...
index created!
creating index...
index created!


### Mask R-CNN ResNet-50 FPN · input 640

In [18]:
run_maskrcnn_r50_fpn_640 = train_maskrcnn_run(640)
show_training_result(run_maskrcnn_r50_fpn_640, "Mask R-CNN ResNet-50 FPN", 640)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 215MB/s]


maskrcnn_r50_fpn_640_resmi epoch 1/30:   0%|          | 0/397 [00:00<?, ?it/s]

Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)


Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 2.499265373353694, 'val_mask_map50_95': 0.03198098887245626, 'val_mask_map50': 0.057729735156350714, 'seconds': 90.35237979888916}


... (84 outputs left out)

maskrcnn_r50_fpn_640_resmi epoch 30/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 30, 'train_loss': 0.29935615841928, 'val_mask_map50_95': 0.163641165639209, 'val_mask_map50': 0.25568125515154966, 'seconds': 86.69736218452454}
ZIP selesai: /kaggle/working/maskrcnn_r50_fpn_640_resmi.zip


,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,Mask R-CNN ResNet-50 FPN,640,30,0.1660,/kaggle/working/visi_komputer_041/maskrcnn_r50_fpn_640_resmi/best.pth


### Mask R-CNN ResNet-50 FPN · input 320

In [19]:
run_maskrcnn_r50_fpn_320 = train_maskrcnn_run(320)
show_training_result(run_maskrcnn_r50_fpn_320, "Mask R-CNN ResNet-50 FPN", 320)

maskrcnn_r50_fpn_320_resmi epoch 1/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 2.7287254045982503, 'val_mask_map50_95': 0.03230928013515745, 'val_mask_map50': 0.057322988192489314, 'seconds': 74.93770670890808}


maskrcnn_r50_fpn_320_resmi epoch 2/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 1.9435252375356495, 'val_mask_map50_95': 0.05417026875215931, 'val_mask_map50': 0.09880543021481698, 'seconds': 73.98861837387085}


... (81 outputs left out)

maskrcnn_r50_fpn_320_resmi epoch 30/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 30, 'train_loss': 0.37189599550716224, 'val_mask_map50_95': 0.12235465809525557, 'val_mask_map50': 0.19777749259745656, 'seconds': 68.06580519676208}
ZIP selesai: /kaggle/working/maskrcnn_r50_fpn_320_resmi.zip


,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,Mask R-CNN ResNet-50 FPN,320,30,0.1224,/kaggle/working/visi_komputer_041/maskrcnn_r50_fpn_320_resmi/best.pth


### Mask R-CNN ResNet-50 FPN · input 960

In [20]:
run_maskrcnn_r50_fpn_960 = train_maskrcnn_run(960)
show_training_result(run_maskrcnn_r50_fpn_960, "Mask R-CNN ResNet-50 FPN", 960)

maskrcnn_r50_fpn_960_resmi epoch 1/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 2.4442460772672887, 'val_mask_map50_95': 0.03008500866973569, 'val_mask_map50': 0.05200912777109294, 'seconds': 122.28777623176575}


maskrcnn_r50_fpn_960_resmi epoch 2/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 2, 'train_loss': 1.7635756393974313, 'val_mask_map50_95': 0.06141307262666674, 'val_mask_map50': 0.10223002626451437, 'seconds': 121.3952124118805}


... (81 outputs left out)

maskrcnn_r50_fpn_960_resmi epoch 30/30:   0%|          | 0/397 [00:00<?, ?it/s]

Validasi mask:   0%|          | 0/166 [00:00<?, ?it/s]

{'epoch': 30, 'train_loss': 0.3027178700028048, 'val_mask_map50_95': 0.16540917617599213, 'val_mask_map50': 0.2470003443998724, 'seconds': 119.82281541824341}
ZIP selesai: /kaggle/working/maskrcnn_r50_fpn_960_resmi.zip


,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,Mask R-CNN ResNet-50 FPN,960,30,0.1719,/kaggle/working/visi_komputer_041/maskrcnn_r50_fpn_960_resmi/best.pth


## Ringkasan sembilan eksperimen

In [21]:
summary = pd.DataFrame(training_summary.values())
summary.to_csv(WORK_DIR / "ringkasan_training.csv", index=False)
display(summary.style.format({"Val mask AP50–95 terbaik": "{:.4f}"}))
print(f"Konfigurasi selesai: {len(summary)}/9")
print("Ringkasan:", WORK_DIR / "ringkasan_training.csv")
print("ZIP tersedia pada sesi ini:")
for archive in sorted(WORK_DIR.glob("*.zip")):
    print(f"{archive.name} ({archive.stat().st_size / 1024**2:.1f} MB)")

,Model,Input,Epoch selesai,Val mask AP50–95 terbaik,Checkpoint
0,YOLOv8n-seg,640,30,0.1426,/kaggle/working/visi_komputer_041/yolov8n_seg_640_resmi/weights/best.pt
1,YOLOv8n-seg,320,30,0.1073,/kaggle/working/visi_komputer_041/yolov8n_seg_320_resmi/weights/best.pt
2,YOLOv8n-seg,960,30,0.1326,/kaggle/working/visi_komputer_041/yolov8n_seg_960_resmi/weights/best.pt
3,YOLOv8s-seg,640,30,0.2209,/kaggle/working/visi_komputer_041/yolov8s_seg_640_resmi/weights/best.pt
4,YOLOv8s-seg,320,30,0.1707,/kaggle/working/visi_komputer_041/yolov8s_seg_320_resmi/weights/best.pt
5,YOLOv8s-seg,960,30,0.2104,/kaggle/working/visi_komputer_041/yolov8s_seg_960_resmi/weights/best.pt
6,Mask R-CNN ResNet-50 FPN,640,30,0.1660,/kaggle/working/visi_komputer_041/maskrcnn_r50_fpn_640_resmi/best.pth
7,Mask R-CNN ResNet-50 FPN,320,30,0.1224,/kaggle/working/visi_komputer_041/maskrcnn_r50_fpn_320_resmi/best.pth
8,Mask R-CNN ResNet-50 FPN,960,30,0.1719,/kaggle/working/visi_komputer_041/maskrcnn_r50_fpn_960_resmi/best.pth


Konfigurasi selesai: 9/9
Ringkasan: /kaggle/working/ringkasan_training.csv
ZIP tersedia pada sesi ini:
maskrcnn_r50_fpn_320_resmi.zip (628.2 MB)
maskrcnn_r50_fpn_640_resmi.zip (627.9 MB)
maskrcnn_r50_fpn_960_resmi.zip (627.9 MB)
yolov8n_seg_320_resmi.zip (17.6 MB)
yolov8n_seg_640_resmi.zip (19.9 MB)
yolov8n_seg_960_resmi.zip (20.4 MB)
yolov8s_seg_320_resmi.zip (48.1 MB)
yolov8s_seg_640_resmi.zip (50.4 MB)
yolov8s_seg_960_resmi.zip (51.0 MB)
